# HW4 bonus: A precision budget for a decision model (10 pts)

Jev-style "System One" models make many small structured decisions, for example routing a customer request to the right team. Each decision is cheap, but the model that makes them still has to be stored and served. This bonus asks how far such a model can be compressed before its decisions change.

The supplied model, Decision-1.0-Kai, uses a ModernBERT encoder to score a request against a list of options and return one probability per option. You will first implement symmetric per-row int8 quantization, then decide how many bits each part of the model receives.

Use a GPU runtime (in Colab: Runtime > Change runtime type > GPU). A CPU also works, more slowly.

## Setup

Run this cell unchanged. It downloads the course support code, checks its integrity, and downloads the model once (about 1.9 GB).

In [ ]:
#@title Setup (run unchanged)
import hashlib, io, sys, urllib.request, zipfile
from pathlib import Path

SUPPORT_URL = "https://duke.box.com/shared/static/2lr382zdrb6ccfnngllkmqyqikvahwb3.zip"
SUPPORT_SHA256 = "b90a625013e737d7089f76c72ef2a728f218e28ca2b43a11b45fccdd7f657291"

root = Path("/content/hw4") if "google.colab" in sys.modules else Path.cwd() / "hw4"
support = root / "support" / SUPPORT_SHA256[:12]
if not (support / "hw4_decision").is_dir():
    data = urllib.request.urlopen(SUPPORT_URL, timeout=300).read()
    if hashlib.sha256(data).hexdigest() != SUPPORT_SHA256:
        raise RuntimeError("The course support package failed its integrity check. Rerun this cell, or contact course staff.")
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        archive.extractall(support)
sys.path.insert(0, str(support))

# The course model adapter is tested with one transformers version. Colab ships another.
import importlib.metadata, subprocess
PINNED = {"transformers": "4.57.6"}
def _installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
missing = [f"{name}=={version}" for name, version in PINNED.items() if _installed(name) != version]
if missing:
    if "transformers" in sys.modules:
        raise RuntimeError("Restart the session (Runtime > Restart session), then run this cell again.")
    if "google.colab" not in sys.modules:
        raise RuntimeError("Install the pinned library in this notebook first by running: %pip install " + " ".join(missing))
    print("Installing the pinned model library once (under a minute).", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import torch
from hw4_decision.bonus import Bonus
hw = Bonus.open(root / "work")

## Watch the model decide

`hw.decide(request, options)` shows how the original model and an int8 copy split their probability across the options you give. Try your own requests and options. Notice that changing the option wording can change the decision.

In [ ]:
hw.decide("I was charged twice for my coffee this morning", ["billing", "login", "returns", "shipping"])
hw.decide("My card stopped working after I moved abroad", ["card not working", "change address", "travel notice", "close account"])
hw.decide("Where is the money I sent yesterday?", ["transfer pending", "refund request", "card delivery"])

## Part (a): Symmetric per-row quantization (4 pts)

Implement `quantize_rows(weight)` for a two-dimensional `torch.float32` tensor. For each row $i$, use the scale $s_i = \max_j |W_{ij}| / 127$. Divide the row by $s_i$, round to the nearest integer, clip to $[-127, 127]$, and convert to `torch.int8`.

Return `(quantized, scale)`, where `quantized` has the input's shape and `scale` is a `torch.float32` tensor of shape `(rows, 1)`. An all-zero row must map to zeros and still receive a positive scale.

Only the cell below is graded, and it runs on its own with only `torch` imported. Keep its first line and its function name, and put any helpers inside it.

In [ ]:
# HW4-GRADED: quantize_rows (keep this first line)
def quantize_rows(weight):
    # TODO: compute one scale per row and return the int8 weights and the scales.
    raise NotImplementedError("Implement the quantization function.")

In [ ]:
hw.check(quantize_rows)

## Part (b): Precision budget (6 pts)

Choose how many bits each part of the model receives:

- `embeddings`: the token embedding table, 61% of the model's weights.
- `attention`: the attention layers.
- `ffn`: the feed-forward layers.

For each group, pick `bits` (2, 3, 4, 6 or 8) and `clip` (0.50 to 1.00, the fraction of each row's largest weight used as the clipping bound). Everything else stays in float32.

Your policy earns points if the compressed model makes the same decision as the original model on at least 95% of the held-out requests. It then earns 3 points at 319 MB or less, or 6 points at 270 MB or less. For reference, the int8 model from part (a) is 368 MB.

The preview below estimates this on 154 public practice requests and lists the decisions that changed. The official score is computed on a Google Cloud TPU after the deadline.

Only the policy cell is graded. Keep its first line, and keep only `policy = {...}` with literal values. The starter policy is plain int8, which earns 0 points here.

In [ ]:
# HW4-GRADED: policy (keep this first line)
policy = {
    "embeddings": {"bits": 8, "clip": 1.00},
    "attention":  {"bits": 8, "clip": 1.00},
    "ffn":        {"bits": 8, "clip": 1.00},
}

In [ ]:
result = hw.preview(policy)

## See your decisions change

Rerun your requests with your policy next to the original and int8 models.

In [ ]:
hw.decide("Where is the money I sent yesterday?", ["transfer pending", "refund request", "card delivery"], policy=policy)

## Submission

Save this notebook and upload the `.ipynb` file to the Gradescope assignment "ECE661 Homework 4 - Bonus (notebook)". Resubmission is allowed until the deadline.

- Part (a) is graded within minutes of each submission.
- Part (b) shows your policy's exact size right away. Its points are posted after the deadline, once every submitted policy has been evaluated on the TPU.

No written report is required for the bonus.